# HuggingFace Pipeline

In [1]:
!pip install -q transformers datasets   # HuggingFace 모델 / 파이프라인 + 데이터셋 유틸
!pip install -q sentencepiece           # 서브워드 토크나이저(SentencePiece)
!pip install -q kobert-transformers     # koBERT 전용 transformer 래퍼
!pip install -q python-dotenv           # .env

In [2]:
import os                           # 운영체제 경로 설정 및 접근
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"]="1"
from dotenv import load_dotenv      # .env 파일을 환경변수로 로드
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')


### NLP Tasks

주어진 task에서 사전 훈련된 모델을 사용하는 가장 간단한 방법은 `pipeline`을 사용하는 것이다.

In [3]:
from transformers import pipeline

pipeline() : 항상 Hugging Face Hub 에 있는 모델을 기본으로 사용
- 모델을 명시하지 않는 경우.  
    - HuggingFace가 task별 기본 (default) 모델을 자동선택
    - 이 기본모델은 HuggingFace Hub에 공개된 모델
- 모델을 명시한 경우
    - Higgingface에 올라온 특정 repo에서 다운로드. 
- 모델 객체를 직접 넘길. 경우.   
    - 로컬에서 로드    
    - 직접 학습/파인튜닝한 모델    
    - 이런 경우에는 pipeline이 "추론래포"의 역할만 한다.

##### 기계번역

https://huggingface.co/Helsinki-NLP/opus-mt-ko-en

In [4]:
translator = pipeline('translation', model='Helsinki-NLP/opus-mt-ko-en')
translator

/Users/jy/NLP/nlp_venv_311/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use mps:0


In [5]:
translator('간식비를 향하여!!!')

[{'translation_text': 'To the snacks!'}]

In [6]:
translator([
    '공부를 열심히 하시면 좋겠어요.',
    '수업을 열심히 들으시면 좋겠고요.',
    '실습도 열심히 해주시면 좋겠어요.',
    '제발 17기 말 좀... 들으세요...'
])

[{'translation_text': 'I hope you study hard.'},
 {'translation_text': "I'd like you to take the class really hard."},
 {'translation_text': "I'd like you to work hard on your practice."},
 {'translation_text': 'Please listen to 17th century...'}]

##### 감성 분석

In [7]:
sentiment_clf = pipeline('sentiment-analysis')

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [8]:
sentiment_clf("I'm very happy!!!")

[{'label': 'POSITIVE', 'score': 0.9998722076416016}]

- 한국어

https://huggingface.co/sangrimlee/bert-base-multilingual-cased-nsmc

In [9]:
ko_sentiment_clf = pipeline('sentiment-analysis', model='sangrimlee/bert-base-multilingual-cased-nsmc')

Device set to use mps:0


In [10]:
ko_sentiment_clf([
    '어제 배운 Attention이 어려워서 맥주를 한 잔 했어.',
    '공부는 어렵지만 맥주는 시원하더라',
    '하지만 나는 굴하지 않고 열심히 공부할거야!!!'
])

[{'label': 'negative', 'score': 0.904799222946167},
 {'label': 'negative', 'score': 0.7013087868690491},
 {'label': 'positive', 'score': 0.7580994963645935}]

##### Zero Shot Classification

- shot == 예시
- 예시 없이 (학습 없이) 추론(분류)

In [11]:
zero_shot_clf = pipeline('zero-shot-classification')

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


In [12]:
zero_shot_clf(
    "This is a course about the transformers library.",
    candidate_labels=["education", "politics", "business"]
)

{'sequence': 'This is a course about the transformers library.',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.9053577184677124, 0.07259666919708252, 0.022045625373721123]}

In [13]:
zero_shot_clf(
    "It has a poison and it's very dangerous",
    candidate_labels=["tiger", "squirrel", "snake"]
)

{'sequence': "It has a poison and it's very dangerous",
 'labels': ['snake', 'tiger', 'squirrel'],
 'scores': [0.6373928189277649, 0.19870348274707794, 0.16390372812747955]}

- 한국어

https://huggingface.co/joeddav/xlm-roberta-large-xnli

In [14]:
ko_zero_shot_clf = pipeline('zero-shot-classification', model='joeddav/xlm-roberta-large-xnli')

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [15]:
sequence_to_classify = "2025년에는 어떤 운동을 하시겠습니까?!"
candidate_labels = ["정치", "경제", "건강"]

ko_zero_shot_clf(sequence_to_classify, candidate_labels)

{'sequence': '2025년에는 어떤 운동을 하시겠습니까?!',
 'labels': ['정치', '건강', '경제'],
 'scores': [0.4039546549320221, 0.3669039309024811, 0.22914138436317444]}

In [16]:
sequence_to_classify = "2025년에는 어떤 운동을 하시겠습니까?!"
candidate_labels = ["정치", "경제", "건강"]
hypothesis_template = "이 텍스트는 {}에 관한 내용입니다."

ko_zero_shot_clf(sequence_to_classify, candidate_labels, hypothesis_template=hypothesis_template)

{'sequence': '2025년에는 어떤 운동을 하시겠습니까?!',
 'labels': ['건강', '정치', '경제'],
 'scores': [0.9117783308029175, 0.059240855276584625, 0.028980819508433342]}

##### 텍스트 생성

In [17]:
text_generator = pipeline('text-generation', model='distilgpt2')

Device set to use mps:0


In [18]:
text_generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2
)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In this course, we will teach you how to build out your own program in the event of a disaster.\n\n\n\n\nWe will teach you how to build a program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nIn this course, we will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the event of a disaster.\nWe will teach you how to build out your own program in the 

- 한국어

https://huggingface.co/skt/ko-gpt-trinity-1.2B-v0.5

In [19]:
ko_text_generator = pipeline('text-generation', model='skt/ko-gpt-trinity-1.2B-v0.5')

Device set to use mps:0


In [20]:
# ko_text_generator('AI란 말입니다')
ko_text_generator('오늘 점심에는')

[{'generated_text': '오늘 점심에는 뭐 먹지\n 답변:바삭한 돈까스 어떠세요?'}]

##### Fill Mask

In [21]:
unmasker = pipeline('fill-mask')

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

/Users/jy/NLP/nlp_venv_311/lib/python3.11/site-packages/huggingface_hub/file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 331.06 MB. The target location /Users/jy/.cache/huggingface/hub/models--distilbert--distilroberta-base/blobs only has 136.98 MB free disk space.
  warnings.warn(
/Users/jy/NLP/nlp_venv_311/lib/python3.11/site-packages/huggingface_hub/file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 331.06 MB. The target location /Users/jy/.cache/huggingface/hub/models--distilbert--distilroberta-base/blobs only has 136.97 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

/Users/jy/NLP/nlp_venv_311/lib/python3.11/site-packages/huggingface_hub/file_download.py:798: UserWarning: Not enough free disk space to download the file. The expected file size is: 331.06 MB. The target location /Users/jy/.cache/huggingface/hub/models--distilbert--distilroberta-base/blobs only has 165.85 MB free disk space.
  warnings.warn(


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 8aa9537c-c3ab-4b3c-884a-3e89a0583c12)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/distilbert/distilroberta-base/fb53ab8802853c8e4fbdbcd0529f21fc6f459b2b/tokenizer.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: beeb5cb4-9c65-46df-b522-5100edc91a84)')' thrown while requesting GET https://huggingface.co/api/resolve-cache/models/distilbert/distilroberta-base/fb53ab8802853c8e4fbdbcd0529f21fc6f459b2b/tokenizer.json
Retrying in 1s [Retry 1/5].


tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use mps:0


In [22]:
unmasker('This course will teach you all about <mask> model.', top_k=3)

[{'score': 0.09073150157928467,
  'token': 265,
  'token_str': ' business',
  'sequence': 'This course will teach you all about business model.'},
 {'score': 0.07262653857469559,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical model.'},
 {'score': 0.0493052639067173,
  'token': 5,
  'token_str': ' the',
  'sequence': 'This course will teach you all about the model.'}]

##### NER

In [23]:
ner = pipeline('ner', model='dslim/bert-base-NER', grouped_entities=True)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use mps:0
/Users/jy/NLP/nlp_venv_311/lib/python3.11/site-packages/transformers/pipelines/token_classification.py:186: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


In [24]:
ner('My name is Squirrel and I work at Playdata in Seoul')

[{'entity_group': 'PER',
  'score': np.float32(0.98995215),
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity_group': 'PER',
  'score': np.float32(0.46767762),
  'word': '##qui',
  'start': 12,
  'end': 15},
 {'entity_group': 'PER',
  'score': np.float32(0.7266252),
  'word': '##rrel',
  'start': 15,
  'end': 19},
 {'entity_group': 'ORG',
  'score': np.float32(0.986338),
  'word': 'Playdata',
  'start': 34,
  'end': 42},
 {'entity_group': 'LOC',
  'score': np.float32(0.9994649),
  'word': 'Seoul',
  'start': 46,
  'end': 51}]

##### Q&A

In [25]:
qna = pipeline('question-answering')

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use mps:0


In [26]:
qna(
    question="Where do I work?",
    context="My name is Squirrel and I work at Playdata in Seoul"
)

{'score': 0.8710195663734339, 'start': 34, 'end': 42, 'answer': 'Playdata'}

- 한국어

In [27]:
ko_qna = pipeline('question-answering', model='klue/roberta-base')

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of RobertaForQuestionAnswering were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Device set to use mps:0


In [28]:
ko_qna(
    question="1931년 어떤 진단을 받았나요?",
    context="""그러던 1931년, 그는 갑작스럽게 폐결핵을 진단받았다. 이상은 하루에 담배를 50개피 이상 피는 것을 자신의 일과라고 표현했을 정도로 엄청난 골초였는데, 그 때문인지 병세는 날이 갈수록 악화되어 담당의가 그의 폐를 확인하고는 형체도 안 보인다며 혀를 내둘렀을 정도였다. 결국 1933년부터는 각혈까지 시작되었고, 건축기사 일을 지속하기 어렵다고 판단한 이상은 조선총독부에서 퇴사하고 황해도에 있는 배천 온천으로 요양을 간다."""
)

{'score': 0.00028987726545892656,
 'start': 44,
 'end': 71,
 'answer': '50개피 이상 피는 것을 자신의 일과라고 표현했을'}

In [29]:
help(ko_qna)

Help on QuestionAnsweringPipeline in module transformers.pipelines.question_answering object:

class QuestionAnsweringPipeline(transformers.pipelines.base.ChunkPipeline)
 |  QuestionAnsweringPipeline(model: Union[ForwardRef('PreTrainedModel'), ForwardRef('TFPreTrainedModel')], tokenizer: transformers.tokenization_utils.PreTrainedTokenizer, modelcard: Optional[transformers.modelcard.ModelCard] = None, framework: Optional[str] = None, task: str = '', **kwargs)
 |  
 |  Question Answering pipeline using any `ModelForQuestionAnswering`. See the [question answering
 |  examples](../task_summary#question-answering) for more information.
 |  
 |  Example:
 |  
 |  ```python
 |  >>> from transformers import pipeline
 |  
 |  >>> oracle = pipeline(model="deepset/roberta-base-squad2")
 |  >>> oracle(question="Where do I live?", context="My name is Wolfgang and I live in Berlin")
 |  {'score': 0.9191, 'start': 34, 'end': 40, 'answer': 'Berlin'}
 |  ```
 |  
 |  Learn more about the basics of usin